# 02 — Feature Engineering

**Objetivo:** Limpeza, encoding, criação de features derivadas e geração dos splits treino/teste.

**Inputs:** `data/raw/dataset.csv`

**Outputs:** `data/processed/features.parquet + splits + artefatos`

---

**Roteiro:**

1. Setup — imports e carregamento
2. Limpeza — nulos, duplicatas, colunas constantes
3. Features derivadas — variáveis construídas a partir das hipóteses do NB01
4. Encoding + Split + Scaling
5. Split treino/teste estratificado
6. Salvamento


In [1]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    LabelEncoder,
    OneHotEncoder,
    OrdinalEncoder,
    StandardScaler,
)
from statsmodels.stats.outliers_influence import (
    variance_inflation_factor,
)

from src.config import CAMINHOS, CONFIG
from src.viz_config import CORES, PALETTE


SEED = CONFIG["dados"]["random_state"]
TARGET = CONFIG["dados"]["target_col"]
TEST_SIZE = CONFIG["dados"]["test_size"]

DATA_DIR = CAMINHOS.dados_raw
PROC_DIR = CAMINHOS.dados_processed
MODELS_DIR = CAMINHOS.modelos
FIG_DIR = CAMINHOS.figures

PROC_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)


print("✅ Setup OK")
print(f"   TARGET     : {TARGET}")
print(f"   PROC_DIR   : {PROC_DIR}")
print(f"   MODELS_DIR : {MODELS_DIR}")

✅ Setup OK
   TARGET     : Attrition
   PROC_DIR   : C:\Users\jas_t\Repo\Portfolios\ibm_attrition\data\processed
   MODELS_DIR : C:\Users\jas_t\Repo\Portfolios\ibm_attrition\models


### Etapa 2 - Carregamento e Limpeza

In [2]:
df = pd.read_csv(DATA_DIR / CAMINHOS.dados_brutos)
print(f'Shape bruto: {df.shape}')
df.head()

Shape bruto: (1470, 35)


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


Com base nas análises exploratórias realizadas no NB01, algumas variáveis foram removidas do dataset por apresentarem baixa relevância preditiva, ausência de variabilidade ou forte redundância informacional.

- As colunas descartadas foram: `EmployeeCount`, `Over18`, `StandardHours`, `EmployeeNumber`, `JobLevel` e `PerformanceRating`.

A remoção dessas features contribui para reduzir redundância, simplificar o modelo e minimizar problemas de multicolinearidade durante a etapa de modelagem.

In [3]:
COLS_DROP = [
    'EmployeeCount',
    'Over18',
    'StandardHours',
    'EmployeeNumber',
    'JobLevel',
    'PerformanceRating'
]

df = df.drop(columns=COLS_DROP)
print(f'Shape após descarte: {df.shape}')
print(f'Colunas descartadas: {COLS_DROP}')

Shape após descarte: (1470, 29)
Colunas descartadas: ['EmployeeCount', 'Over18', 'StandardHours', 'EmployeeNumber', 'JobLevel', 'PerformanceRating']


Vamos converter a variável `Attrition` para binário: Yes → 1, No → 0

In [4]:
df[TARGET] = (df[TARGET] == 'Yes').astype(int)
print(f'\nTarget convertido → Yes=1: {df[TARGET].sum()} | No=0: {(df[TARGET]==0).sum()}')


Target convertido → Yes=1: 237 | No=0: 1233


In [5]:
print(f"=== Validações ===")
assert df.isnull().sum().sum() == 0, 'Nulos encontrados!'
assert df.duplicated().sum() == 0,   'Duplicatas encontradas!'
print('\n✅ Validações OK — zero nulos, zero duplicatas')
print(f'Shape final limpo: {df.shape}')

=== Validações ===

✅ Validações OK — zero nulos, zero duplicatas
Shape final limpo: (1470, 29)


### Etapa 3 - Features Derivadas

A partir dos insights obtidos no NB01, foi definida a criação de novas features derivadas com o objetivo de aumentar o poder preditivo do modelo e representar padrões de comportamento identificados durante a análise exploratória.

As novas variáveis foram desenvolvidas especialmente para capturar perfis associados a maior risco de *attrition*, incluindo:

- funcionários jovens com baixa experiência profissional;
- colaboradores com longos períodos sem promoção;
- possíveis sinais de estagnação de carreira;
- combinações entre tempo de empresa, cargo e progressão profissional.

Além das features contínuas derivadas, também foram planejadas *flags* binárias para representar perfis específicos de risco, facilitando a interpretação dos modelos e permitindo capturar padrões que variáveis isoladas não conseguem representar diretamente.

Essa estratégia busca transformar os insights obtidos na EDA em variáveis mais informativas para os algoritmos de Machine Learning.

In [6]:
df['IncomePerYear'] = df['MonthlyIncome'] / (df['TotalWorkingYears'] + 1)

df['YearsPerCompany'] = df['TotalWorkingYears'] / (df['NumCompaniesWorked'] + 1)

df['SatisfacaoMedia'] = df[[
    'JobSatisfaction', 'EnvironmentSatisfaction',
    'RelationshipSatisfaction', 'WorkLifeBalance'
]].mean(axis=1)

df['PromocaoAtrasada'] = (df['YearsSinceLastPromotion'] > 3).astype(int)

df['JovemSemExperiencia'] = ((df['Age'] < 30) & (df['TotalWorkingYears'] < 3)).astype(int)

NOVAS = ['IncomePerYear', 'YearsPerCompany', 'SatisfacaoMedia',
         'PromocaoAtrasada', 'JovemSemExperiencia']

print(f'Features criadas: {NOVAS}\n')
print(f'Shape após feature engineering: {df.shape}')

Features criadas: ['IncomePerYear', 'YearsPerCompany', 'SatisfacaoMedia', 'PromocaoAtrasada', 'JovemSemExperiencia']

Shape após feature engineering: (1470, 34)


In [7]:
df.head(8)

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,Gender,...,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,IncomePerYear,YearsPerCompany,SatisfacaoMedia,PromocaoAtrasada,JovemSemExperiencia
0,41,1,Travel_Rarely,1102,Sales,1,2,Life Sciences,2,Female,...,1,6,4,0,5,665.888889,0.888889,2.00,0,0
1,49,0,Travel_Frequently,279,Research & Development,8,1,Life Sciences,3,Male,...,3,10,7,1,7,466.363636,5.000000,3.00,0,0
2,37,1,Travel_Rarely,1373,Research & Development,2,2,Other,4,Male,...,3,0,0,0,0,261.250000,1.000000,3.00,0,0
3,33,0,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,4,Female,...,3,8,7,3,0,323.222222,4.000000,3.25,0,0
4,27,0,Travel_Rarely,591,Research & Development,2,1,Medical,1,Male,...,3,2,2,2,2,495.428571,0.600000,2.50,0,0
5,32,0,Travel_Frequently,1005,Research & Development,2,2,Life Sciences,4,Male,...,2,7,7,3,6,340.888889,8.000000,3.25,0,0
6,59,0,Travel_Rarely,1324,Research & Development,3,3,Medical,3,Female,...,2,1,0,0,0,205.384615,2.400000,1.75,0,0
7,30,0,Travel_Rarely,1358,Research & Development,24,1,Life Sciences,4,Male,...,3,1,0,0,0,1346.500000,0.500000,3.00,0,0


In [8]:
print(f"=== Distrição das flags ===")
print(f"\nPromocaoAtrasada   → 1: {df['PromocaoAtrasada'].sum()} ({df['PromocaoAtrasada'].mean()*100:.1f}%)")
print(f"JovemSemExperiencia → 1: {df['JovemSemExperiencia'].sum()} ({df['JovemSemExperiencia'].mean()*100:.1f}%)")

=== Distrição das flags ===

PromocaoAtrasada   → 1: 321 (21.8%)
JovemSemExperiencia → 1: 83 (5.6%)


- `IncomePerYear`: varia bastante (205 a 1346 nas primeiras linhas) — captura bem a relação salário/experiência.
- `PromocaoAtrasada`: 21.8% dos funcionários sem promoção há mais de 3 anos — grupo relevante para investigar no SHAP.
- `JovemSemExperiencia`: 5.6% — grupo pequeno mas de alto risco conforme a EDA mostrou.

In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 34 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Age                       1470 non-null   int64  
 1   Attrition                 1470 non-null   int64  
 2   BusinessTravel            1470 non-null   str    
 3   DailyRate                 1470 non-null   int64  
 4   Department                1470 non-null   str    
 5   DistanceFromHome          1470 non-null   int64  
 6   Education                 1470 non-null   int64  
 7   EducationField            1470 non-null   str    
 8   EnvironmentSatisfaction   1470 non-null   int64  
 9   Gender                    1470 non-null   str    
 10  HourlyRate                1470 non-null   int64  
 11  JobInvolvement            1470 non-null   int64  
 12  JobRole                   1470 non-null   str    
 13  JobSatisfaction           1470 non-null   int64  
 14  MaritalStatus      

### Etapa 4 - Encoding + Split + Scaling

Primeiramente, vamos separar as features por tipo de transformação que será aplicado:

- Features binárias -> `OrdinalEncoder`
- Features nominais -> `OneHotEncoder`
- Features numéricas com distribuições próximas à normal -> `StandardScaler`
- Features sem transformação (Ordinais numéricas e Flags) -> `passthrough`

Vamos também remover do pipeline as features com alta corelação detectada no NB01

- Features para remover: `SatisfacaoMedia`, `YearsInCurrentRole` e `YearsWithCurrManager`

In [10]:
ORDINAL_COLS   = ['Gender', 'OverTime']
ORDINAL_ORDERS = [['Female','Male'], ['No','Yes']]

NOMINAL_COLS = ['BusinessTravel', 'Department', 'EducationField', 'JobRole', 'MaritalStatus']

STANDARD_COLS = [
    'Age', 'DailyRate', 'DistanceFromHome', 'HourlyRate', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked',
    'PercentSalaryHike', 'TotalWorkingYears', 'TrainingTimesLastYear', 'YearsAtCompany',
    'YearsSinceLastPromotion', 'IncomePerYear', 'YearsPerCompany',
]

PASSTHROUGH_COLS = [
    'Education', 'EnvironmentSatisfaction', 'JobInvolvement', 'JobSatisfaction',
    'RelationshipSatisfaction', 'StockOptionLevel', 'WorkLifeBalance',
    'PromocaoAtrasada', 'JovemSemExperiencia',
]

DROP_COLS = ['SatisfacaoMedia', 'YearsInCurrentRole', 'YearsWithCurrManager']

Para garantir a consistência do processo, iremos criar uma feature composta a partir da combinação das variáveis definidas anteriormente e, em seguida, verificar a existência de registros duplicados no dataset.

In [11]:
FEATURES_CT = (ORDINAL_COLS + NOMINAL_COLS + STANDARD_COLS + PASSTHROUGH_COLS)

assert len(FEATURES_CT) == len(set(FEATURES_CT)), \
    f'Duplicatas: {[f for f in FEATURES_CT if FEATURES_CT.count(f)>1]}'
print(f'✅ Sem duplicatas — {len(FEATURES_CT)} features input')

✅ Sem duplicatas — 30 features input


Inicialmente, o dataset foi dividido entre:

- X: conjunto de variáveis preditoras (features);
- y: variável alvo (Attrition).

Em seguida, os dados foram separados em conjuntos de treino e teste utilizando a função train_test_split() da biblioteca scikit-learn, gerando:

- X_train e y_train para treinamento dos modelos;
- X_test e y_test para avaliação de desempenho.

Durante essa etapa, foi aplicada a estratificação da variável alvo (stratify=y), garantindo que a proporção entre as classes fosse preservada em ambos os conjuntos. Essa abordagem é especialmente importante em problemas com classes desbalanceadas, como o presente caso, evitando distorções na distribuição do target entre treino e teste e proporcionando avaliações mais confiáveis dos modelos.

In [12]:
X = df[FEATURES_CT]
y = df[TARGET]

print(f'\nShape X : {X.shape}')
print(f'Shape y : {y.shape}')


Shape X : (1470, 30)
Shape y : (1470,)


In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=CONFIG["dados"]["test_size"],
    random_state=SEED,
    stratify=y
)

print(f'\nX_train : {X_train.shape}')
print(f'X_test  : {X_test.shape}')
print(f'Taxa Attrition treino : {y_train.mean():.3f}')
print(f'Taxa Attrition teste  : {y_test.mean():.3f}')
print(f'Taxa resposta treino: {y_train.mean()*100:.1f}%')


X_train : (1176, 30)
X_test  : (294, 30)
Taxa Attrition treino : 0.162
Taxa Attrition teste  : 0.160
Taxa resposta treino: 16.2%


Nesta etapa de pré-processamento, foi utilizado o ColumnTransformer da biblioteca scikit-learn para aplicar transformações específicas às variáveis de acordo com suas características estatísticas e tipos de dados.

Com o objetivo de otimizar o desempenho dos diferentes algoritmos de Machine Learning, foram criados dois preprocessors distintos:

- um pipeline voltado para modelos tradicionais/sensíveis à escala;
- outro pipeline específico para algoritmos tree-based.

In [14]:
preprocessamento_arvore = ColumnTransformer(
    transformers=[
        ('ordinal', OrdinalEncoder(categories=ORDINAL_ORDERS, handle_unknown='use_encoded_value', unknown_value=-1), ORDINAL_COLS),
        ('nominal', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), NOMINAL_COLS),
        ('passthrough', 'passthrough', STANDARD_COLS + PASSTHROUGH_COLS),
    ],
    remainder='drop',
    verbose_feature_names_out=True
)

preprocessor = ColumnTransformer(
    transformers=[
        ("ordinal", OrdinalEncoder(categories=ORDINAL_ORDERS, handle_unknown="use_encoded_value", unknown_value=-1), ORDINAL_COLS),
        ("nominal", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False), NOMINAL_COLS),
        ("standard", StandardScaler(), STANDARD_COLS),
        ("passthrough", "passthrough", PASSTHROUGH_COLS),
    ],
    remainder="drop",
    verbose_feature_names_out=True
)

Após a definição do ColumnTransformer, foi criada uma pipeline de pré-processamento utilizando os recursos da biblioteca scikit-learn.

Essa pipeline encapsula todas as etapas de transformação dos dados em uma única estrutura, permitindo executar o fluxo completo de preparação de forma padronizada e reproduzível.

Na sequência:

- o pipeline foi ajustado ao conjunto de treino utilizando fit_transform();
- o mesmo pipeline foi aplicado ao conjunto de teste utilizando transform().

In [15]:
pipeline_linear = Pipeline([('preprocessor', preprocessor)])
pipeline_arvore = Pipeline([('preprocessor', preprocessamento_arvore)])

X_train_s = pipeline_linear.fit_transform(X_train)
X_test_s  = pipeline_linear.transform(X_test)

Após a aplicação das etapas de pré-processamento, os dados transformados foram convertidos novamente para o formato DataFrame. Essa etapa teve como objetivo preservar a interpretabilidade do pipeline, mantendo os nomes das features geradas após a etapa de preprocessamento.

In [16]:
feature_names_out = list(pipeline_linear.named_steps['preprocessor'].get_feature_names_out())

X_train_s = pd.DataFrame(X_train_s, columns=feature_names_out)
X_test_s  = pd.DataFrame(X_test_s,  columns=feature_names_out)

In [17]:
print(f'\n✅ Dois preprocessadores criados:')
print(f'   pipeline_linear : {X_train_s.shape[1]} features '
      f'(com StandardScaler → LR, SVC, KNN)')
print(f'   pipeline_arvore : sem StandardScaler '
      f'(→ DT, LGBM, XGB)')

ohe_cols = [c for c in feature_names_out if 'nominal__' in c]
print(f'\nColunas OHE geradas: {len(ohe_cols)}')
for c in ohe_cols:
    print(f'   → {c}')


✅ Dois preprocessadores criados:
   pipeline_linear : 44 features (com StandardScaler → LR, SVC, KNN)
   pipeline_arvore : sem StandardScaler (→ DT, LGBM, XGB)

Colunas OHE geradas: 19
   → nominal__BusinessTravel_Travel_Frequently
   → nominal__BusinessTravel_Travel_Rarely
   → nominal__Department_Research & Development
   → nominal__Department_Sales
   → nominal__EducationField_Life Sciences
   → nominal__EducationField_Marketing
   → nominal__EducationField_Medical
   → nominal__EducationField_Other
   → nominal__EducationField_Technical Degree
   → nominal__JobRole_Human Resources
   → nominal__JobRole_Laboratory Technician
   → nominal__JobRole_Manager
   → nominal__JobRole_Manufacturing Director
   → nominal__JobRole_Research Director
   → nominal__JobRole_Research Scientist
   → nominal__JobRole_Sales Executive
   → nominal__JobRole_Sales Representative
   → nominal__MaritalStatus_Married
   → nominal__MaritalStatus_Single


In [18]:
pipeline_linear

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('ordinal', ...), ('nominal', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers conta

### Etapa 5 - Salvamento

Após a conclusão das etapas de pré-processamento, os principais artefatos do pipeline foram salvos para garantir reprodutibilidade, organização do projeto e reutilização nas próximas etapas de modelagem. Os arquivos persistidos incluem:

- conjuntos de treino e teste processados/escalonados;
- versões raw dos splits (X_train_raw, X_test_raw, y_train e y_test), preservando os dados sem transformações;
- dataset completo após o pré-processamento;
- pipeline de transformação e demais artefatos utilizados durante o processamento.

In [19]:
## 5. Salvamento de artefatos
import joblib

PROC_DIR  = CAMINHOS.dados_processed
MODEL_DIR = CAMINHOS.modelos
PROC_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# ── Splits escalados ──────────────────────────────────────────
X_train_s.to_parquet(PROC_DIR / 'X_treino.parquet', index=False)
X_test_s.to_parquet(PROC_DIR  / 'X_teste.parquet',  index=False)
y_train.to_frame().reset_index(drop=True).to_parquet(
    PROC_DIR / 'y_treino.parquet', index=False)
y_test.to_frame().reset_index(drop=True).to_parquet(
    PROC_DIR / 'y_teste.parquet',  index=False)

# Splits RAW — para Pipeline completo nos NB03/NB04
X_train.reset_index(drop=True).to_parquet(
    PROC_DIR / 'X_treino_raw.parquet', index=False)
X_test.reset_index(drop=True).to_parquet(
    PROC_DIR / 'X_teste_raw.parquet',  index=False)

# Dataset completo processado
df.to_parquet(PROC_DIR / 'features.parquet', index=False)

# ── Artefatos ─────────────────────────────────────────────────
joblib.dump(pipeline_linear, MODEL_DIR / 'pipeline_linear.pkl')
joblib.dump(pipeline_arvore, MODEL_DIR / 'pipeline_arvore.pkl')
joblib.dump(FEATURES_CT,            MODEL_DIR / 'feature_names_input.pkl')
joblib.dump(feature_names_out,      MODEL_DIR / 'feature_names_output.pkl')

# ── Verificação ───────────────────────────────────────────────
arquivos = (list(PROC_DIR.glob('*.parquet')) +
            list(MODEL_DIR.glob('*.pkl')))

print('✅ NB02 — Feature Engineering (Pipeline) finalizado.')
print(f'\nArquivos salvos:')
for a in sorted(arquivos):
    print(f'   → {a.parent.name}/{a.name}  '
          f'({a.stat().st_size/1024:.1f} KB)')

print(f'\n=== Resumo ===')
print(f'Features input          : {len(FEATURES_CT)}')
print(f'Features após encoding  : {X_train_s.shape[1]}')
print(f'X_train                 : {X_train_s.shape}')
print(f'X_test                  : {X_test_s.shape}')
print(f'Taxa Attrition treino   : {y_train.mean():.3f}\n')
print('   → pipeline_linear.pkl (LR, SVC, KNN)')
print('   → pipeline_arvore.pkl (DT, LGBM, XGB)')

✅ NB02 — Feature Engineering (Pipeline) finalizado.

Arquivos salvos:
   → processed/features.parquet  (71.2 KB)
   → processed/X_teste.parquet  (48.7 KB)
   → processed/X_teste_raw.parquet  (29.8 KB)
   → processed/X_treino.parquet  (88.5 KB)
   → processed/X_treino_raw.parquet  (57.2 KB)
   → processed/y_teste.parquet  (1.3 KB)
   → processed/y_treino.parquet  (1.4 KB)
   → models/feature_names_input.pkl  (0.5 KB)
   → models/feature_names_output.pkl  (1.4 KB)
   → models/pipeline_arvore.pkl  (1.3 KB)
   → models/pipeline_linear.pkl  (7.2 KB)

=== Resumo ===
Features input          : 30
Features após encoding  : 44
X_train                 : (1176, 44)
X_test                  : (294, 44)
Taxa Attrition treino   : 0.162

   → pipeline_linear.pkl (LR, SVC, KNN)
   → pipeline_arvore.pkl (DT, LGBM, XGB)


### Resumo do NB02

| Etapas | Entregável |
|---|---|
| 1 | Setup — `CONFIG`, `viz_config` e `paths` |
| 2 | Limpeza — 6 features removidas |
| 3 | 5 features derivadas — e distribuição de flags |
| 4 | Pipeline `sklearn` — `OrdinalEncoder` + `OneHotEncoder` + `PowerTransformer` + `RobustScaler` + `StandardScaler` |
| 5 | 7 arquivos `.parquet` + 4 arquivos `.pkl` salvos |